# Notebook 4 - Feature Engineering
Generate ML features from market data.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("processed/master_market_data.parquet")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Symbol","Date"]).reset_index(drop=True)
df.head()


## Return Features

In [ ]:
df["Daily_Return"] = df.groupby("Symbol")["Close"].pct_change()
df["Log_Return"] = np.log(df["Close"]/df.groupby("Symbol")["Close"].shift(1))


## Moving Averages

In [ ]:
for w in [5,20,50]:
    df[f"SMA_{w}"] = df.groupby("Symbol")["Close"].transform(lambda s: s.rolling(w).mean())

df["EMA_12"] = df.groupby("Symbol")["Close"].transform(lambda s: s.ewm(span=12,adjust=False).mean())
df["EMA_26"] = df.groupby("Symbol")["Close"].transform(lambda s: s.ewm(span=26,adjust=False).mean())


## MACD

In [ ]:
df["MACD"] = df["EMA_12"] - df["EMA_26"]
df["MACD_Signal"] = df.groupby("Symbol")["MACD"].transform(lambda s: s.ewm(span=9,adjust=False).mean())
df["MACD_Histogram"] = df["MACD"] - df["MACD_Signal"]


## RSI (14)

In [ ]:
delta = df.groupby("Symbol")["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.groupby(df["Symbol"]).transform(lambda s: s.rolling(14).mean())
avg_loss = loss.groupby(df["Symbol"]).transform(lambda s: s.rolling(14).mean())
rs = avg_gain/avg_loss
df["RSI"] = 100 - (100/(1+rs))


## Bollinger Bands

In [ ]:
ma20 = df.groupby("Symbol")["Close"].transform(lambda s: s.rolling(20).mean())
std20 = df.groupby("Symbol")["Close"].transform(lambda s: s.rolling(20).std())
df["BB_Middle"] = ma20
df["BB_Upper"] = ma20 + 2*std20
df["BB_Lower"] = ma20 - 2*std20


## Volatility & Volume

In [ ]:
df["Rolling_Volatility"] = df.groupby("Symbol")["Daily_Return"].transform(lambda s: s.rolling(20).std())
df["Volume_Change"] = df.groupby("Symbol")["Volume"].pct_change()
df["Volume_MA_5"] = df.groupby("Symbol")["Volume"].transform(lambda s: s.rolling(5).mean())
df["Volume_MA_20"] = df.groupby("Symbol")["Volume"].transform(lambda s: s.rolling(20).mean())


## Lag Features

In [ ]:
for i in [1,2,3]:
    df[f"Lag_Close_{i}"] = df.groupby("Symbol")["Close"].shift(i)
df["Lag_Return_1"] = df.groupby("Symbol")["Daily_Return"].shift(1)


## Target Variables

In [ ]:
df["Next_Close"] = df.groupby("Symbol")["Close"].shift(-1)
df["Next_Return"] = df.groupby("Symbol")["Daily_Return"].shift(-1)
df["Price_Direction"] = (df["Next_Close"]>df["Close"]).astype(int)


## Save

In [ ]:
df.to_parquet("processed/feature_engineered_data.parquet",index=False)
df.to_csv("processed/feature_engineered_data.csv",index=False)
print(df.head())
print("Saved feature_engineered_data.parquet and feature_engineered_data.csv")
